# Lesson 8: GPU training for real

Goes with the Lesson 8 page of the Forkcast course. Devices, mixed precision, augmentation, checkpoints and resume, timing and logging: training loop v2.

Runtime: pick a **GPU**. Every cell also runs on a CPU (autocast falls back to bfloat16), just slower.

## 1. Clone your repository and install the package

In [ ]:
GITHUB_USER = "your-github-username"   # edit this line
REPO = "forkcast-course"

!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!pip install -e .

## 2. Meet the device

In [ ]:
import importlib, time, torch
from torch import nn
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from forkcast import trips, train, vision
from forkcast.model import TabularNet
from forkcast.preprocess import Normalizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if device.type == "cuda" else "(no GPU: everything below still runs)")
x = torch.randn(1000, 1000); y = x.to(device)
print("tensor .to() copies:", x.device, "->", y.device)
model = TabularNet(4).to(device)
print("module .to() moves in place:", next(model.parameters()).device)

frame = trips.load_trips(verbose=False)
train_frame, val_frame, _ = trips.split_frame(frame, seed=0)
norm = Normalizer().fit(torch.tensor(train_frame[trips.CONTINUOUS].values, dtype=torch.float32))
train_loader, val_loader = trips.make_loaders(train_frame, val_frame, norm, batch_size=512)
cont, cat, minutes = next(iter(train_loader))
try:
    model(cont, cat)                                     # batch still on the CPU
except RuntimeError as err:
    print("RuntimeError:", str(err)[:110])

## 3. `forkcast/train.py`: training loop v2

Fill in `make_scaler`, `train_one_epoch_amp`, `save_checkpoint`, `load_checkpoint` and `fit_v2` as the page builds them. Re-run the cell after each edit.

In [ ]:
%%writefile forkcast/train.py
"""Lesson 3. The training toolkit: one epoch, evaluation, fit, save, load (reference solution)."""

from __future__ import annotations

from pathlib import Path

import torch
from torch import Tensor, nn


def train_one_epoch(model: nn.Module, X: Tensor, y: Tensor, optimizer: torch.optim.Optimizer, loss_fn: nn.Module) -> float:
    """One pass over the data. Returns the loss as a Python float."""
    model.train()
    optimizer.zero_grad()                 # 1. clean slate (Lesson 2's zero_() for every parameter)
    loss = loss_fn(model(X), y)           #    forward
    loss.backward()                       # 2. slopes into every parameter's .grad
    optimizer.step()                      # 3. the update rule, applied to every parameter
    return loss.item()


@torch.no_grad()
def evaluate(model: nn.Module, X: Tensor, y: Tensor, loss_fn: nn.Module) -> float:
    """Loss on data the model is not learning from."""
    model.eval()
    return loss_fn(model(X), y).item()


def fit(model: nn.Module, X: Tensor, y: Tensor, epochs: int = 300, lr: float = 0.1,
        optimizer_cls=torch.optim.SGD, loss_fn: nn.Module | None = None) -> list[float]:
    """Build the optimizer, run ``epochs`` epochs, return one loss per epoch."""
    loss_fn = loss_fn or nn.MSELoss()
    optimizer = optimizer_cls(model.parameters(), lr=lr)
    return [train_one_epoch(model, X, y, optimizer, loss_fn) for _ in range(epochs)]


def save(model: nn.Module, path: str | Path) -> None:
    torch.save(model.state_dict(), path)


def load(model: nn.Module, path: str | Path) -> nn.Module:
    model.load_state_dict(torch.load(path, weights_only=True))
    return model


# ---- Lesson 5: validation, early stopping, regularization ------------------

def fit_with_validation(model: nn.Module, X_train: Tensor, y_train: Tensor, X_val: Tensor, y_val: Tensor,
                        epochs: int = 300, lr: float = 0.01, optimizer_cls=torch.optim.Adam,
                        weight_decay: float = 0.0, patience: int | None = None, loss_fn: nn.Module | None = None) -> dict:
    """Train while watching the validation loss; keep the best weights; optionally stop early."""
    import copy

    loss_fn = loss_fn or nn.MSELoss()
    optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train": [], "val": [], "best_epoch": 0, "best_val": float("inf")}
    best_state = copy.deepcopy(model.state_dict())
    since_best = 0
    for epoch in range(epochs):
        history["train"].append(train_one_epoch(model, X_train, y_train, optimizer, loss_fn))
        val_loss = evaluate(model, X_val, y_val, loss_fn)
        history["val"].append(val_loss)
        if val_loss < history["best_val"]:
            history["best_val"], history["best_epoch"] = val_loss, epoch
            best_state = copy.deepcopy(model.state_dict())     # a snapshot, not a reference
            since_best = 0
        else:
            since_best += 1
            if patience is not None and since_best >= patience:
                break                                           # early stopping
    model.load_state_dict(best_state)                           # hand back the best model, not the last one
    return history


# ---- Lesson 6: the toolkit learns to take a DataLoader ------------------------

def _split_batch(batch, device):
    """A batch is a tuple (input_1, ..., input_k, target). Move everything to ``device``."""
    *inputs, target = batch
    return [x.to(device) for x in inputs], target.to(device)


def train_one_epoch_loader(model: nn.Module, loader, optimizer: torch.optim.Optimizer, loss_fn: nn.Module, device: str | torch.device = "cpu") -> float:
    """One pass over every batch in ``loader``. Returns the mean training loss, weighted by batch size.

    For each batch: split it, zero_grad, forward with ``model(*inputs)``, loss, backward, step.
    """
    model.train()
    total, seen = 0.0, 0
    for batch in loader:
        inputs, target = _split_batch(batch, device)
        optimizer.zero_grad()
        loss = loss_fn(model(*inputs), target)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(target)          # weight by batch size: the last batch may be smaller
        seen += len(target)
    return total / seen


@torch.no_grad()
def evaluate_loader(model: nn.Module, loader, loss_fn: nn.Module, device: str | torch.device = "cpu") -> float:
    """Mean loss over every batch in ``loader``, weighted by batch size, in eval mode."""
    model.eval()
    total, seen = 0.0, 0
    for batch in loader:
        inputs, target = _split_batch(batch, device)
        total += loss_fn(model(*inputs), target).item() * len(target)
        seen += len(target)
    return total / seen


def fit_loader(model: nn.Module, train_loader, val_loader, epochs: int = 20, lr: float = 1e-3, optimizer_cls=torch.optim.Adam,
               weight_decay: float = 0.0, patience: int | None = None, loss_fn: nn.Module | None = None, device: str | torch.device = "cpu") -> dict:
    """Lesson 5's fit_with_validation, with loaders instead of tensors. Same history, same early stopping, same best-weights rule."""
    import copy

    model.to(device)
    loss_fn = loss_fn or nn.MSELoss()
    optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train": [], "val": [], "best_epoch": 0, "best_val": float("inf")}
    best_state = copy.deepcopy(model.state_dict())
    since_best = 0
    for epoch in range(epochs):
        history["train"].append(train_one_epoch_loader(model, train_loader, optimizer, loss_fn, device))
        val_loss = evaluate_loader(model, val_loader, loss_fn, device)
        history["val"].append(val_loss)
        if val_loss < history["best_val"]:
            history["best_val"], history["best_epoch"] = val_loss, epoch
            best_state = copy.deepcopy(model.state_dict())
            since_best = 0
        else:
            since_best += 1
            if patience is not None and since_best >= patience:
                break
    model.load_state_dict(best_state)
    return history


# ---- Lesson 8: devices, mixed precision, checkpoints, logging: training loop v2 ----

import time


def autocast_dtype(device: str | torch.device) -> torch.dtype:
    """float16 on a GPU (fast tensor cores), bfloat16 on a CPU (same range as float32, no scaler needed)."""
    return torch.float16 if torch.device(device).type == "cuda" else torch.bfloat16


def make_scaler(device: str | torch.device, amp: bool):
    """A GradScaler only where it helps: float16 on CUDA. Return None otherwise.

    float16 cannot hold gradients smaller than about 6e-8; the scaler multiplies the loss by a
    large factor before backward so they survive, then divides the gradients back before the step.
    bfloat16 (CPU) has float32's range, so no scaling is needed there.
    """
    raise NotImplementedError("Lesson 8, step 4: torch.amp.GradScaler(\"cuda\") if amp and the device is cuda, else None")


def train_one_epoch_amp(model: nn.Module, loader, optimizer: torch.optim.Optimizer, loss_fn: nn.Module,
                        device: str | torch.device = "cpu", amp: bool = False, scaler=None) -> float:
    """Lesson 6's epoch with two additions: autocast around the forward pass, and a scaler when given.

    Compute the forward pass inside ``torch.autocast(device_type=device.type, dtype=autocast_dtype(device), enabled=amp)``,
    then the loss *outside* it in float32 (``model_output.float()``). With a scaler:
    ``scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()``; without: backward and step as usual.
    Use ``optimizer.zero_grad(set_to_none=True)``. Return the mean loss weighted by batch size.
    """
    raise NotImplementedError("Lesson 8, step 4: the Lesson 6 loop with autocast and the optional scaler")


def save_checkpoint(path: str | Path, model: nn.Module, optimizer: torch.optim.Optimizer, epoch: int, history: dict, best_val: float) -> None:
    """Everything needed to continue training as if nothing happened: model, optimizer, epoch, history, best_val."""
    raise NotImplementedError("Lesson 8, step 6: torch.save({\"model\": ..., \"optimizer\": ..., \"epoch\": epoch, \"history\": history, \"best_val\": best_val}, path)")


def load_checkpoint(path: str | Path, model: nn.Module, optimizer: torch.optim.Optimizer | None = None, device: str | torch.device = "cpu") -> dict:
    """Load a checkpoint into ``model`` (and ``optimizer`` if given) and return the whole dict.

    Use ``torch.load(path, map_location=device, weights_only=True)``: a checkpoint saved on a GPU must open on a CPU.
    """
    raise NotImplementedError("Lesson 8, step 6: ckpt = torch.load(...); model.load_state_dict(ckpt[\"model\"]); optimizer too; return ckpt")


def fit_v2(model: nn.Module, train_loader, val_loader, epochs: int = 10, lr: float = 1e-3, device: str | torch.device = "cpu",
           amp: bool = False, loss_fn: nn.Module | None = None, weight_decay: float = 0.0, patience: int | None = None,
           ckpt_path: str | Path | None = None, resume: bool = False, log_dir: str | Path | None = None, seed: int = 0) -> dict:
    """Training loop v2. Lesson 6's fit_loader plus: device placement, optional autocast, a checkpoint
    after every epoch, resuming from it, per-epoch timing, and optional TensorBoard logging.

    The history gains an "epoch_seconds" list. Seed the RNG with ``seed + epoch`` at the start of
    every epoch, so a resumed run shuffles exactly like an uninterrupted one. When ``resume`` is True
    and ``ckpt_path`` exists, load it and continue from ``epoch + 1`` with its history.
    """
    raise NotImplementedError("Lesson 8, step 7: fit_loader with device, amp, checkpointing, resume, timing and logging")

### Time an epoch on each device

In [ ]:
importlib.reload(train)
shape_train, shape_val = vision.shape_loaders(1000, 400)
for dev in ["cpu"] + (["cuda"] if torch.cuda.is_available() else []):
    torch.manual_seed(0)
    h = train.fit_v2(vision.SmallCNN(), shape_train, shape_val, epochs=2, lr=1e-3, loss_fn=nn.CrossEntropyLoss(), device=dev)
    print(f"{dev}: {h['epoch_seconds'][1]:.2f} s per epoch")

### Mixed precision

In [ ]:
importlib.reload(train)
print("scaler on", device, ":", train.make_scaler(device, amp=True))
def fresh():
    torch.manual_seed(0); return TabularNet(4, hidden=(32,))
with torch.autocast(device_type=device.type, dtype=train.autocast_dtype(device)):
    print("output dtype under autocast:", fresh().to(device)(cont.to(device), cat.to(device)).dtype)
fp32 = train.fit_v2(fresh(), train_loader, val_loader, epochs=3, lr=3e-3, device=device)
amp = train.fit_v2(fresh(), train_loader, val_loader, epochs=3, lr=3e-3, device=device, amp=True)
print("float32:", [round(v, 2) for v in fp32["train"]], "| mixed:", [round(v, 2) for v in amp["train"]])
print("seconds per epoch: float32", round(fp32["epoch_seconds"][-1], 3), "| mixed", round(amp["epoch_seconds"][-1], 3))

## 4. Augmentation: `forkcast/vision.py`

`train_transform` is provided; write `AugmentedTensorDataset`.

In [ ]:
%%writefile forkcast/vision.py
"""Lesson 7. Images as tensors, and the first convolutional network.

``make_shapes`` (provided) draws synthetic pictures of four shapes so the tests
run offline; the notebook trains the same network on real dish photos.
"""

from __future__ import annotations

import math

import torch
import torch.nn.functional as F
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset, TensorDataset

SHAPE_CLASSES = ["circle", "square", "triangle", "cross"]


def make_shapes(n: int = 1000, seed: int = 0, size: int = 32) -> tuple[Tensor, Tensor]:
    """Return ``(images, labels)``: images (n, 3, size, size) float32 in [0, 1], labels (n,) int64.

    Each image is one coloured shape on a dark, lightly noisy background, at a random
    position and scale. Classes in SHAPE_CLASSES order: circle, square, triangle, cross.
    """
    g = torch.Generator().manual_seed(seed)
    labels = torch.randint(0, 4, (n,), generator=g)
    yy, xx = torch.meshgrid(torch.arange(size, dtype=torch.float32), torch.arange(size, dtype=torch.float32), indexing="ij")
    images = torch.zeros(n, 3, size, size)
    for i in range(n):
        cx, cy = (size * (0.3 + 0.4 * torch.rand(2, generator=g))).tolist()
        r = size * (0.12 + 0.13 * torch.rand(1, generator=g).item())
        dx, dy = xx - cx, yy - cy
        k = labels[i].item()
        if k == 0:
            mask = dx ** 2 + dy ** 2 <= r ** 2
        elif k == 1:
            mask = (dx.abs() <= r) & (dy.abs() <= r)
        elif k == 2:
            mask = (dy >= -r) & (dy <= r) & (dx.abs() <= (r - dy) * 0.6)
        else:
            mask = ((dx.abs() <= r * 0.3) & (dy.abs() <= r)) | ((dy.abs() <= r * 0.3) & (dx.abs() <= r))
        colour = 0.5 + 0.5 * torch.rand(3, generator=g)
        images[i] = 0.08 * torch.rand(3, size, size, generator=g)
        images[i][:, mask] = colour[:, None]
    return images.clamp(0, 1), labels


def shape_loaders(n_train: int = 1000, n_val: int = 400, batch_size: int = 64, size: int = 32) -> tuple[DataLoader, DataLoader]:
    """Loaders over TensorDatasets of synthetic shapes (train seed 0, validation seed 1)."""
    x_tr, y_tr = make_shapes(n_train, seed=0, size=size)
    x_val, y_val = make_shapes(n_val, seed=1, size=size)
    return (DataLoader(TensorDataset(x_tr, y_tr), batch_size=batch_size, shuffle=True, drop_last=True),
            DataLoader(TensorDataset(x_val, y_val), batch_size=batch_size, shuffle=False))


def conv_output_size(size: int, kernel: int, stride: int = 1, padding: int = 0) -> int:
    """Spatial size after a convolution or pooling layer: floor((size + 2p - k) / s) + 1."""
    return (size + 2 * padding - kernel) // stride + 1


def accuracy(logits: Tensor, y: Tensor) -> float:
    """Fraction of rows whose highest logit is the true class. (N, C), (N,) -> float."""
    return (logits.argmax(dim=1) == y).float().mean().item()


def cross_entropy_by_hand(logits: Tensor, y: Tensor) -> Tensor:
    """What nn.CrossEntropyLoss computes: minus the log-probability of the true class, averaged.

    Use F.log_softmax over the class dimension, pick each row's true-class entry, negate, mean.
    """
    logp = F.log_softmax(logits, dim=1)                       # (N, C) log-probabilities, rows sum to 1 in probability
    return -logp[torch.arange(len(y)), y].mean()               # pick each row's true class


class SmallCNN(nn.Module):
    """Two conv blocks, then a small classifier head.

    Block: Conv2d(3x3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool2d(2). Channels 3 -> 16 -> 32.
    After two pools a size x size image is (size/4) x (size/4), so the head's first
    Linear takes 32 * (size // 4) ** 2 inputs. The head ends in Linear(64, n_classes), no activation:
    those are the logits.
    """

    def __init__(self, n_classes: int = 4, size: int = 32):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),   # (N, 16, size/2, size/2)
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),  # (N, 32, size/4, size/4)
        )
        self.head = nn.Sequential(
            nn.Flatten(),                                          # (N, 32 * (size/4)^2)
            nn.Linear(32 * (size // 4) ** 2, 64), nn.ReLU(),
            nn.Linear(64, n_classes),                              # logits, no activation
        )

    def forward(self, x: Tensor) -> Tensor:
        """(N, 3, size, size) -> (N, n_classes)"""
        return self.head(self.features(x))


@torch.no_grad()
def accuracy_loader(model: nn.Module, loader, device: str | torch.device = "cpu") -> float:
    """Accuracy over every batch in ``loader``."""
    model.eval()
    correct, seen = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(dim=1) == y).sum().item()
        seen += len(y)
    return correct / seen


# ---- Lesson 8: data augmentation --------------------------------------------

def train_transform(size: int = 32):
    """Random changes that keep the label true: a flipped, slightly re-framed, re-lit dish is the same dish."""
    from torchvision.transforms import v2
    return v2.Compose([
        v2.RandomHorizontalFlip(),
        v2.RandomResizedCrop(size, scale=(0.7, 1.0), antialias=True),
        v2.ColorJitter(brightness=0.2, contrast=0.2),
    ])


class AugmentedTensorDataset(Dataset):
    """Like TensorDataset(images, labels) but applies ``transform`` to the image every time an item is fetched.

    A fresh random transform per fetch means the model never sees exactly the same picture twice.
    With ``transform=None`` it must return items identical to the stored tensors.
    """

    def __init__(self, images: Tensor, labels: Tensor, transform=None):
        raise NotImplementedError("Lesson 8, step 5: store images, labels, transform")

    def __len__(self) -> int:
        raise NotImplementedError("Lesson 8, step 5")

    def __getitem__(self, i: int):
        raise NotImplementedError("Lesson 8, step 5: x = self.images[i]; apply the transform if there is one; return x, self.labels[i]")


def augmented_shape_loaders(n_train: int = 1000, n_val: int = 400, batch_size: int = 64, size: int = 32) -> tuple[DataLoader, DataLoader]:
    """Training loader with augmentation, validation loader without (never augment what you evaluate on)."""
    x_tr, y_tr = make_shapes(n_train, seed=0, size=size)
    x_val, y_val = make_shapes(n_val, seed=1, size=size)
    return (DataLoader(AugmentedTensorDataset(x_tr, y_tr, train_transform(size)), batch_size=batch_size, shuffle=True, drop_last=True),
            DataLoader(TensorDataset(x_val, y_val), batch_size=batch_size, shuffle=False))

In [ ]:
importlib.reload(vision)
images, labels = vision.make_shapes(8, seed=0)
aug = vision.AugmentedTensorDataset(images, labels, vision.train_transform(32))
print("two fetches equal?", torch.equal(aug[0][0], aug[0][0]))
plt.figure(figsize=(10, 1.6)); plt.axis("off")
plt.imshow(make_grid(torch.stack([aug[0][0] for _ in range(8)]), nrow=8).permute(1, 2, 0))

small_train, small_val = vision.shape_loaders(200, 400, batch_size=32)
small_aug, _ = vision.augmented_shape_loaders(200, 400, batch_size=32)
torch.manual_seed(0); plain = vision.SmallCNN(); train.fit_v2(plain, small_train, small_val, epochs=15, lr=1e-3, loss_fn=nn.CrossEntropyLoss(), device=device)
torch.manual_seed(0); augm = vision.SmallCNN(); train.fit_v2(augm, small_aug, small_val, epochs=15, lr=1e-3, loss_fn=nn.CrossEntropyLoss(), device=device)
print("200 shapes, 15 epochs: plain", round(vision.accuracy_loader(plain, small_val, device), 3), "| augmented", round(vision.accuracy_loader(augm, small_val, device), 3))

## 5. Checkpoints: resume is exact

In [ ]:
importlib.reload(train)
full = train.fit_v2(fresh(), train_loader, val_loader, epochs=5, lr=3e-3, ckpt_path="full.pt")
train.fit_v2(fresh(), train_loader, val_loader, epochs=3, lr=3e-3, ckpt_path="r.pt")                 # "crash" after epoch 2
resumed = train.fit_v2(fresh(), train_loader, val_loader, epochs=5, lr=3e-3, ckpt_path="r.pt", resume=True)
print("uninterrupted:", [round(v, 2) for v in full["train"]])
print("resumed:      ", [round(v, 2) for v in resumed["train"]])
print(sorted(torch.load("full.pt", weights_only=True).keys()))

### Break it on purpose: weights only

In [ ]:
train.fit_v2(fresh(), train_loader, val_loader, epochs=3, lr=3e-3, ckpt_path="r2.pt")
ckpt = torch.load("r2.pt", weights_only=True)
ckpt["optimizer"] = torch.optim.Adam(fresh().parameters(), lr=3e-3).state_dict()     # throw Adam's memory away
torch.save(ckpt, "r2.pt")
weights_only = train.fit_v2(fresh(), train_loader, val_loader, epochs=5, lr=3e-3, ckpt_path="r2.pt", resume=True)
print("weights only: ", [round(v, 2) for v in weights_only["train"]])

## 6. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs
h = train.fit_v2(TabularNet(4), train_loader, val_loader, epochs=20, lr=3e-3, device=device, amp=True, log_dir="runs/tabular")
print("best validation", round(h["best_val"], 2), "at epoch", h["best_epoch"], "|", round(sum(h["epoch_seconds"]), 1), "s total")

## 7. Definition of done

Expected: `7 passed`, then `62 passed` with no failures for everything so far (skipped tests belong to lessons you have not started).

In [ ]:
!pytest tests/test_08.py -q

In [ ]:
!pytest -m fast -q

## 8. Commit and push

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token with access to your repo: ")

!git config user.name "{GITHUB_USER}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
!git add -A
!git commit -m "Lesson 8: training loop v2"
!git push https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:main